# Step 4 — `rewritten_clean`: clean passages rewritten in LLM style

**Purpose** (protocol §6). PoisonedRAG's poisoned passages were written by an LLM; the clean passages are human-written. A detector could learn "sounds like an LLM = poisoned". This notebook rewrites each test clean passage with an LLM, keeping its facts and length, so the evaluation can test that shortcut: does the detector's false-alarm rate rise on clean passages that only *sound* generated?

**What it does**
1. Reads `data/to_rewrite.jsonl` (the 700 clean passages of the 140 test questions' n=0 groups, written by notebook 02).
2. Rewrites each with `Qwen/Qwen2.5-3B-Instruct` (the Step 1 generator), fp16, greedy decoding, one fixed prompt.
3. **Automatic checks** (fixed before running): non-empty, no preamble, every number in the original still present, length within 0.7–1.3× the original. Passages that fail are **excluded** and counted, never hand-fixed.
4. Writes `data/rewritten_clean.jsonl` (`passage_id`, `text`) for the passing rewrites, a report, and a **20-passage spot-check sheet** (seed 42) for a person to read.

**Then:** re-run `02_build_dataset` (adds the rewritten groups) and `03_distilbert` (scores only the new rows).

**Runtime.** Colab **T4 GPU**, about 30–45 minutes. Progress is saved after every batch, so a disconnect resumes.

## 1. Setup

In [1]:
import os, sys, json, re, time, random, subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print(gpu or "NO GPU — switch the runtime to T4 GPU (Runtime -> Change runtime type)")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "accelerate", "transformers>=4.45,<5"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/rag_trust"
else:
    DRIVE_ROOT = os.environ["RAG_TRUST_ROOT"]
DATA_DIR = f"{DRIVE_ROOT}/data"
SRC = f"{DATA_DIR}/to_rewrite.jsonl"
RAW = f"{DATA_DIR}/rewritten_clean_raw.jsonl"           # every round-1 generation, resumable
RAW2 = f"{DATA_DIR}/rewritten_clean_raw_r2.jsonl"       # round-2 generations (round-1 failures only), resumable
OUT = f"{DATA_DIR}/rewritten_clean.jsonl"                # passing rewrites only (read by notebook 02)
assert os.path.exists(SRC), "run notebooks/02_build_dataset.ipynb first"

CONFIG = {
    "generator": "Qwen/Qwen2.5-3B-Instruct", "dtype": "float16", "decoding": "greedy",
    "batch_size": 8, "max_new_tokens_factor": 1.6, "max_new_tokens_cap": 512,
    "length_ratio_range": [0.7, 1.3], "spot_check_n": 20, "seed": 42,
    "prompt": ("Rewrite the following passage in your own fluent, natural prose. "
               "Keep every fact, name, number and date exactly as it is. Do not add, remove or "
               "explain anything. Keep it about the same length (about {n_words} words). "
               "Reply with the rewritten passage only.\n\nPassage:\n{text}"),
    # Round 2 (added 20 Oct after round 1, see decisions.md): only for passages whose round-1 rewrite failed the checks
    "prompt_round2": ("Paraphrase the passage below sentence by sentence. For each sentence of the original, write one "
                      "sentence that says the same thing in different words. Keep every name, number and date exactly as "
                      "written, using digits wherever the original uses digits. Do not merge, drop or add sentences, do not "
                      "add information, and do not finish or correct the passage. Your answer must be about {n_words} words "
                      "long (between {lo} and {hi} words). Reply with the paraphrase only.\n\nPassage:\n{text}"),
    "check_version": 2,
}
json.dump(CONFIG, open(f"{DATA_DIR}/rewrite_config.json", "w"), indent=2)
ITEMS = [json.loads(l) for l in open(SRC)]
print(f"{len(ITEMS)} passages to rewrite")

GPU 0: Tesla T4 (UUID: GPU-dd96c806-8640-df49-1af5-afb63df0b0bb)
Mounted at /content/drive
700 passages to rewrite


## 2. Automatic checks (version 2)
Version 1 of the number check counted list markers ("2. Purchase…") and Wikipedia citation markers ("[34]") as facts, so it rejected faithful rewrites; it also never noticed numbers the model *added*. Version 2 (decided 20 Oct after reading the round-1 failures; logged in decisions.md) ignores those markers, accepts a digit written out as a word ("9 seasons" → "nine seasons"), and **also rejects any rewrite that adds a number**. The length rule (0.7–1.3×) and the preamble rule are unchanged. Version 2 is applied to both rounds.

In [2]:
NUM = re.compile(r"\d+(?:[.,]\d+)*")
CITE = re.compile(r"\[\d+\]")                                            # citation markers, e.g. [34]
LIST = re.compile(r"(^|[.:;!?]\s+|\s{2,})\d{1,2}(?=[.)]\s|\s{2,}|\s[A-Z])")   # list markers: "2. Purchase", ".  1 I'm"
WORDNUM = {w: str(i) for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen "
           "fourteen fifteen sixteen seventeen eighteen nineteen twenty".split())}
WORDRE = re.compile(r"\b(" + "|".join(WORDNUM) + r")\b", re.I)
PREAMBLE = re.compile(r"^(here is|here's|sure|certainly|rewritten passage|the rewritten)", re.I)

def n_words(s): return len(str(s).split())

def clean_output(s):
    s = s.strip().strip('"').strip()
    s = re.sub(r"^(rewritten passage|passage|paraphrase)\s*:\s*", "", s, flags=re.I)
    return " ".join(s.split())

def numbers(s, words=True):
    s = CITE.sub(" ", s); s = LIST.sub(r"\1", s)
    if words:
        s = WORDRE.sub(lambda m: WORDNUM[m.group(1).lower()], s)
    return {x.replace(",", "") for x in NUM.findall(s)}

def check(original, rewrite):
    """Returns a list of failed checks (empty = passes)."""
    if not rewrite:
        return ["empty"]
    fails = []
    if PREAMBLE.search(rewrite):
        fails.append("preamble")
    lo, hi = CONFIG["length_ratio_range"]
    ratio = n_words(rewrite) / max(n_words(original), 1)
    if not lo <= ratio <= hi:
        fails.append(f"length_ratio_{ratio:.2f}")
    dropped = numbers(original, words=False) - numbers(rewrite)      # every digit in the original survives
    added = numbers(rewrite, words=False) - numbers(original)        # every digit in the rewrite comes from the original
    if dropped:
        fails.append("numbers_dropped:" + ",".join(sorted(dropped)[:5]))
    if added:
        fails.append("numbers_added:" + ",".join(sorted(added)[:5]))
    return fails

assert check("In 1999 it had 3,000 staff [12].", "It employed 3000 people in 1999.") == []
assert check("Trivia.  1 I'm From New Jersey, 13 times.", "I'm From New Jersey, 13 times.") == []
assert check("after 9 seasons it ended", "after nine seasons it ended") == []
assert check("In 1999 it had 3,000 staff.", "It employed many people in 1999.")[0].startswith("numbers_dropped")
assert check("It ran for ten years.", "It ran for 12 years.")[0].startswith("numbers_added")
assert check("one two three four five six seven eight nine ten", "one two")[0].startswith("length_ratio")
assert check("a b c d e f g h i j", "Here is the passage: a b c d e f") == ["preamble"]
print("checks (version 2): ok")

checks (version 2): ok


## 3. Round 1 — rewrite every passage (resumable)

In [3]:
_MODEL = {}
def load_model():
    if "m" not in _MODEL:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(CONFIG["generator"], padding_side="left")
        try:
            model = AutoModelForCausalLM.from_pretrained(CONFIG["generator"], dtype=torch.float16, device_map="auto")
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(CONFIG["generator"], torch_dtype=torch.float16, device_map="auto")
        model.eval(); _MODEL["m"], _MODEL["t"] = model, tok
    return _MODEL["m"], _MODEL["t"]

def generate_batch(batch, prompt):
    if os.environ.get("REWRITE_STUB") == "1":            # offline dry run only: a trivial "rewrite"
        return ["In other words, " + x["text"] for x in batch]
    import torch
    model, tok = load_model()
    lo, hi = CONFIG["length_ratio_range"]
    texts = [tok.apply_chat_template([{"role": "user", "content": prompt.format(
                 n_words=n_words(x["text"]), lo=int(lo * n_words(x["text"]) + 0.999), hi=int(hi * n_words(x["text"])),
                 text=x["text"])}], add_generation_prompt=True, tokenize=False) for x in batch]
    enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
    longest = max(len(tok(x["text"]).input_ids) for x in batch)
    max_new = min(int(CONFIG["max_new_tokens_factor"] * longest) + 20, CONFIG["max_new_tokens_cap"])
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, temperature=None, top_p=None,
                             top_k=None, pad_token_id=tok.eos_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

def run_round(items, path, prompt, label):
    done = {}
    if os.path.exists(path):
        for l in open(path):
            r = json.loads(l); done[r["passage_id"]] = r
    todo = [x for x in items if x["passage_id"] not in done]
    print(f"{label}: {len(done)} already rewritten, {len(todo)} to go")
    t0, bs = time.time(), CONFIG["batch_size"]
    for i in range(0, len(todo), bs):
        batch = todo[i:i + bs]
        outs = generate_batch(batch, prompt)
        with open(path, "a") as f:
            for x, o in zip(batch, outs):
                rec = {"passage_id": x["passage_id"], "dataset": x["dataset"], "qid": x["qid"],
                       "original": x["text"], "rewrite": clean_output(o)}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); done[x["passage_id"]] = rec
        if (i // bs) % 10 == 0:
            print(f"  {min(i + bs, len(todo))}/{len(todo)} ({time.time() - t0:.0f}s)")
    for r in done.values():                                # (re)apply the current checks to every rewrite
        r["fails"] = check(r["original"], r["rewrite"])
    return done

R1 = run_round(ITEMS, RAW, CONFIG["prompt"], "round 1")
r1_fail = [x for x in ITEMS if R1[x["passage_id"]]["fails"]]
print(f"round 1: {len(ITEMS) - len(r1_fail)} pass, {len(r1_fail)} fail (checks v2)")

round 1: 700 already rewritten, 0 to go
round 1: 456 pass, 244 fail (checks v2)


## 3b. Round 2 — regenerate the round-1 failures once, with a sentence-by-sentence prompt
Only passages whose round-1 rewrite failed are regenerated, once, with greedy decoding and the fixed `prompt_round2`. The checks are the same as in round 1. A passage that fails both rounds is excluded (never hand-fixed).

In [4]:
R2 = run_round(r1_fail, RAW2, CONFIG["prompt_round2"], "round 2")
print(f"round 2: {sum(not R2[x['passage_id']]['fails'] for x in r1_fail)} of {len(r1_fail)} now pass")

round 2: 232 already rewritten, 12 to go


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  8/12 (125s)
round 2: 124 of 244 now pass


## 4. Keep the passing rewrites, write the report and the spot-check sheet

In [5]:
import collections
recs = []                                                        # fixed order; round 1 if it passes, else round 2
for x in ITEMS:
    r1 = R1[x["passage_id"]]
    r = dict(r1, round=1) if not r1["fails"] else dict(R2[x["passage_id"]], round=2)
    recs.append(r)
passing = [r for r in recs if not r["fails"]]
with open(OUT, "w") as f:
    for r in passing:
        f.write(json.dumps({"passage_id": r["passage_id"], "text": r["rewrite"]}, ensure_ascii=False) + "\n")

fail_kinds = collections.Counter(("length_ratio" if k.startswith("length_ratio") else k.split(":")[0]) for r in recs for k in r["fails"])
v1_pass = None
if os.path.exists(RAW):                                           # for the record: round 1 under the version-1 check
    v1_pass = sum(1 for l in open(RAW) if not json.loads(l).get("fails", ["?"]))
ratios = [n_words(r["rewrite"]) / max(n_words(r["original"]), 1) for r in passing]
REPORT = {"total": len(recs), "passing": len(passing), "excluded": len(recs) - len(passing),
          "passing_round1": sum(r["round"] == 1 for r in passing), "passing_round2": sum(r["round"] == 2 for r in passing),
          "round1_passing_under_check_v1": v1_pass,
          "excluded_by_check": dict(fail_kinds),
          "passing_by_dataset": dict(collections.Counter(r["dataset"] for r in passing)),
          "length_ratio_passing": {"median": round(float(sorted(ratios)[len(ratios) // 2]), 3) if ratios else None}}
json.dump(REPORT, open(f"{DATA_DIR}/rewrite_report.json", "w"), indent=2)
print(json.dumps(REPORT, indent=2))

# 20-passage spot-check sheet (seed 42) for a person to read: did any fact change?
import csv
rng = random.Random(CONFIG["seed"])
sample = rng.sample(passing, min(CONFIG["spot_check_n"], len(passing)))
with open(f"{DATA_DIR}/rewrite_spotcheck.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["passage_id", "dataset", "round", "original", "rewrite", "verdict (ok / fact_changed / other)", "note"])
    for r in sample:
        w.writerow([r["passage_id"], r["dataset"], r["round"], r["original"], r["rewrite"], "", ""])
print(f"spot-check sheet: {len(sample)} pairs -> data/rewrite_spotcheck.csv")

{
  "total": 700,
  "passing": 580,
  "excluded": 120,
  "passing_round1": 456,
  "passing_round2": 124,
  "round1_passing_under_check_v1": 390,
  "excluded_by_check": {
    "numbers_dropped": 64,
    "length_ratio": 65,
    "numbers_added": 19
  },
  "passing_by_dataset": {
    "hotpotqa": 91,
    "msmarco": 394,
    "nq": 95
  },
  "length_ratio_passing": {
    "median": 0.857
  }
}
spot-check sheet: 20 pairs -> data/rewrite_spotcheck.csv


## 5. What to check, then next steps

1. **Report:** `passing` should be well above 400 (round 1 under check v2 alone gives about 456). Note how many were excluded and why; they go in the report as a limitation, and are never fixed by hand.
2. **Spot-check (≈20 min, a person):** open `data/rewrite_spotcheck.csv` in Google Sheets (File → Import, untick "convert text to numbers"), read each pair, and fill in `verdict`: `ok`, `fact_changed` (any name, number, date or claim differs) or `other`. Report the count; if several facts changed, say so in the limitations.
3. **Re-run `02_build_dataset`** (CPU): it prints `rewritten passages: N` and adds the groups; `protocol checks: all passed`.
4. **Re-run `03_distilbert`** (T4): it scores only the new rows with the saved models; thresholds stay unchanged.
5. **Commit:** this notebook, and from Drive `data/`: `rewritten_clean.jsonl`, `rewrite_report.json`, `rewrite_config.json`, the filled-in `rewrite_spotcheck.csv`, and the rebuilt `passages.jsonl.gz` and `build_report.json`. Also commit `rewritten_clean_raw.jsonl` and `rewritten_clean_raw_r2.jsonl`: together they record every generation and every exclusion.